## Fase 4: Análisis Estadístico y Densidad Léxica

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">4.1. Considerando dos grupos de comentarios (odio y no odio) ¿Cuáles son los 100 lemas más repetidos en los comentarios de cada grupo.</span>

In [30]:
from collections import Counter
from tabulate import tabulate

lemas_sin_odio = Counter() # Diccionario especializado para contar
lemas_con_odio = Counter() # Diccionario especializado para contar

num_filas = sub_data.shape[0]

for i in range(num_filas):

    if sub_data.loc[i, "TIPO DE MENSAJE"] != "COMENTARIO":
        continue

    texto = str(sub_data.loc[i, "CONTENIDO A ANALIZAR"])
    doc = nlp(texto)

    intensidad = sub_data.loc[i, "INTENSIDAD"]

    if intensidad == 0:
        contador_lemas = lemas_sin_odio
    elif intensidad > 0:
        contador_lemas = lemas_con_odio
    else:
        continue

    for token in doc:
        if token.is_alpha and not token.is_stop:
            contador_lemas[token.lemma_.lower()] += 1

top_100_sin_odio = lemas_sin_odio.most_common(100)
top_100_con_odio = lemas_con_odio.most_common(100)


In [31]:
# Mostramos los resultados en tablas
tabla =[]

total_top100_sin_odio = sum(frec for _, frec in top_100_sin_odio)
total_top100_con_odio = sum(frec for _, frec in top_100_con_odio)

for i in range(min(len(top_100_sin_odio), len(top_100_con_odio))):
    lema_s, freq_s = top_100_sin_odio[i]
    lema_c, freq_c = top_100_con_odio[i]

    frel_s = freq_s / total_top100_sin_odio
    frel_c = freq_c / total_top100_con_odio

    tabla.append([lema_s, freq_s, f"{frel_s*100:.2f}%", lema_c, freq_c, f"{frel_c*100:.2f}%"])
    

In [32]:
# Guardamos el archivo de frecuencias en un csv
import os

output_dir = "../reports/tables"
os.makedirs(output_dir, exist_ok=True)

df_frecuencias = pd.DataFrame(tabla, columns=[
    "lema_sin_odio", "frec_sin_odio", "frel_sin_odio", 
    "lema_con_odio", "frec_con_odio", "frel_con_odio"
])

# Usamos index=False para no añadir una columna de IDs innecesaria
# Usamos encoding='utf-8-sig' para que Excel lo abra bien con tildes y eñes
csv_path = os.path.join(output_dir, "analisis_frecuencias_top100.csv")
df_frecuencias.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f"Dataset de frecuencias exportado para uso técnico en: {csv_path}")

Dataset de frecuencias exportado para uso técnico en: ../reports/tables\analisis_frecuencias_top100.csv


In [33]:
print(tabulate(
    tabla,
    headers=["Lema sin odio", "Frecuencia sin odio", "Frecuencia relativa sin odio", "Lema con odio", "Frecuencia con odio", "Frecuencia relativa con odio"],
    tablefmt="github"
))

| Lema sin odio    |   Frecuencia sin odio | Frecuencia relativa sin odio   | Lema con odio    |   Frecuencia con odio | Frecuencia relativa con odio   |
|------------------|-----------------------|--------------------------------|------------------|-----------------------|--------------------------------|
| usuarioofuscado  |                  4475 | 5.80%                          | usuarioofuscado  |                    89 | 6.10%                          |
| año              |                  2277 | 2.95%                          | mierda           |                    60 | 4.11%                          |
| gobierno         |                  2007 | 2.60%                          | puta             |                    49 | 3.36%                          |
| persona          |                  1679 | 2.17%                          | hijo             |                    35 | 2.40%                          |
| españa           |                  1572 | 2.04%                          

<hr>
En comentarios sin odio:

- predominan palabras neutrales e informativos: gobierno, año, persona, país, pademia, público....
- vocabulario más descriptivo y contextual


En comentarios con odio:

- predominan insultos y términos peyorativos: mierda, puta, asco, basura, tonto, gentuza, gilipollas, fascista, terrorista...
- vocabulario más agresivo

### <span style="font-size: 14pt; font-weight: bold; color: #0098cd;">4.2. Comparando ambas listas de los 100 lemas más repetidos ¿Cuántos lemas de la lista de odio no aparecen en la lista sin odio?</span>

In [34]:
lemas_no_comunes = []

lemas1 = {lema for lema, _ in top_100_sin_odio}
lemas2 = {lema for lema, _ in top_100_con_odio}

for lema in lemas2:
    if lema not in lemas1:
        lemas_no_comunes.append(lema)

# Alternativa más simplificada:
# lemas_comunes = {
#     lema: lemas_sin_odio[lema]
#     for lema in lemas1 & lemas2
# }

# Transformamos el conjunto de palabras para que sea más fácil de leer
tabla_lemas = [lemas_no_comunes[i:i+11] for i in range(0, len(lemas_no_comunes), 11)]

# Imprimimos los resultados
print(f"Número total de lemas de comentarios con odio que no aparecen en el otro grupo: {len(lemas_no_comunes)}")
print(tabulate(tabla_lemas, tablefmt="github"))

Número total de lemas de comentarios con odio que no aparecen en el otro grupo: 55
|--------------|------------|-----------|-----------|------------|----------|------------|-------------|------------|-----------|----------|
| miserable    | mentiroso  | corrupto  | hipócrita | psicópata  | importar | democracia | ignorante   | pobre      | d         | culo     |
| inútil       | terrorista | basura    | jeta      | socialista | inepto   | tonto      | informativo | estupidez  | asqueroso | ladrón   |
| malo         | gilipol él | dais      | cárcel    | impuesto   | hijo     | madre      | mierda      | menudo     | único     | política |
| sinvergüenza | morir      | fascista  | favor     | cara       | hp       | asesino    | puto        | tanto      | gentuza   | panfleto |
| asco         | puta       | vergüenza | idiota    | facha      | derecha  | humano     | hdp         | periodismo | comunista | vox      |


In [35]:
# Guardamos el archivo de frecuencias únicas en un csv

output_dir = "../reports/tables"
os.makedirs(output_dir, exist_ok=True)

df_exclusivos = pd.DataFrame(lemas_no_comunes, columns=["lema_exclusivo_odio"])

csv_path_exclusivos = os.path.join(output_dir, "lemas_exclusivos_odio.csv")
df_exclusivos.to_csv(csv_path_exclusivos, index=False, encoding='utf-8-sig')

print(f"Archivo de términos exclusivos guardado en: {csv_path_exclusivos}")

Archivo de términos exclusivos guardado en: ../reports/tables\lemas_exclusivos_odio.csv


In [36]:
# Imprimimos los resultados
print(f"Número total de lemas de comentarios con odio que no aparecen en el otro grupo: {len(lemas_no_comunes)}")
print(tabulate(tabla_lemas, tablefmt="github"))

Número total de lemas de comentarios con odio que no aparecen en el otro grupo: 55
|--------------|------------|-----------|-----------|------------|----------|------------|-------------|------------|-----------|----------|
| miserable    | mentiroso  | corrupto  | hipócrita | psicópata  | importar | democracia | ignorante   | pobre      | d         | culo     |
| inútil       | terrorista | basura    | jeta      | socialista | inepto   | tonto      | informativo | estupidez  | asqueroso | ladrón   |
| malo         | gilipol él | dais      | cárcel    | impuesto   | hijo     | madre      | mierda      | menudo     | único     | política |
| sinvergüenza | morir      | fascista  | favor     | cara       | hp       | asesino    | puto        | tanto      | gentuza   | panfleto |
| asco         | puta       | vergüenza | idiota    | facha      | derecha  | humano     | hdp         | periodismo | comunista | vox      |


<hr>
La comparación sugiere que más de la mitad de los 100 primeros lemas más frecuentes en los comentarios con odio son propios de este grupo, lo que indica que existe un vocabulario característico.